# Lab 2 NLP — Analisis de Sentimientos (Sentiment140)

Notebook principal. **Se ejecuta en SageMaker** (Notebook Instance) para que los runs
queden con `ResourceArn` valido. Ejecutar localmente solo sirve para probar con una
muestra pequena antes de subir a SageMaker (ver CLAUDE.md, seccion "Flujo de trabajo").

## 0. Setup y conexion a MLflow

In [ ]:
import json
import os
import shutil
import warnings

import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")

# ---- Configuracion de MLflow ----
# Reemplazar por la URL publica del MLflow Tracking Server levantado en la EC2.
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://<EC2_PUBLIC_IP>:5000")
EXPERIMENT_NAME = "nlp-lab2-sentiment140"

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
print("Tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT_NAME)

In [ ]:
# ---- Identidad del integrante que ejecuta este notebook ----
# PENDIENTE: confirmar member_id con el profesor antes de ejecutar cualquier run.
LAB_MEMBER_ID = "TODO_CONFIRMAR_CON_PROFESOR"  # ej: "member_01"

# ---- Metadata de procedencia SageMaker (obligatorio en cada run) ----
SAGEMAKER_METADATA_PATH = "/opt/ml/metadata/resource-metadata.json"
PROVENANCE_DIR = "provenance"
os.makedirs(PROVENANCE_DIR, exist_ok=True)


def get_notebook_arn():
    """Lee el ResourceArn del Notebook Instance de SageMaker. No editar el archivo fuente."""
    if not os.path.exists(SAGEMAKER_METADATA_PATH):
        raise FileNotFoundError(
            f"No se encontro {SAGEMAKER_METADATA_PATH}. "
            "Este notebook debe ejecutarse desde una SageMaker Notebook Instance."
        )
    with open(SAGEMAKER_METADATA_PATH) as f:
        metadata = json.load(f)
    return metadata["ResourceArn"]


def log_provenance():
    """Copia el metadata de SageMaker sin editarlo y lo registra como artefacto MLflow."""
    dest = os.path.join(PROVENANCE_DIR, "sagemaker-resource-metadata.json")
    shutil.copy(SAGEMAKER_METADATA_PATH, dest)
    mlflow.log_artifact(dest, artifact_path="provenance")
    return dest


# NOTEBOOK_ARN se resuelve al ejecutar en SageMaker; en local queda en None.
try:
    NOTEBOOK_ARN = get_notebook_arn()
except FileNotFoundError:
    NOTEBOOK_ARN = None
    print("Aviso: no se detecto entorno SageMaker. Los runs generados aqui NO seran validos "
          "para la entrega; usar solo para pruebas locales con muestra reducida.")

print("notebook_arn:", NOTEBOOK_ARN)

## 1. Cargar dataset y generar muestra/folds

**Esta seccion se ejecuta UNA SOLA VEZ entre todo el equipo.** El resultado
(`protocol/partitions.csv`) no se vuelve a generar ni a modificar despues.
Si el archivo ya existe (fue generado por otro integrante y esta en el repo),
saltar la generacion y solo cargarlo.

In [ ]:
DATASET_ID = "adilbekovich/Sentiment140Twitter"
DATASET_REVISION = "b6037e127257d95b9b23d31f78b264b9ebe697fd"

SAMPLE_SIZE = 200_000
RANDOM_SEED = 42
CV_FOLDS = 3

PARTITIONS_PATH = "protocol/partitions.csv"
os.makedirs("protocol", exist_ok=True)

In [ ]:
# Para pruebas locales rapidas, usar LOCAL_SMOKE_TEST=True y un tamano de muestra chico.
# En SageMaker (ejecucion oficial) usar LOCAL_SMOKE_TEST=False para respetar el protocolo real.
LOCAL_SMOKE_TEST = NOTEBOOK_ARN is None
SMOKE_TEST_ROWS = 1_000

from datasets import load_dataset

ds = load_dataset(DATASET_ID, revision=DATASET_REVISION)
# train: 1_360_000 registros | test: 240_000 registros
# 0 -> negative, 1 -> positive
print(ds)

In [ ]:
train_df = ds["train"].to_pandas()
train_df["original_index"] = train_df.index  # indice original del split train (desde 0)

if LOCAL_SMOKE_TEST:
    print(f"[SMOKE TEST] Usando solo {SMOKE_TEST_ROWS} filas para prueba local.")
    train_df_work = train_df.sample(
        n=SMOKE_TEST_ROWS, stratify=train_df["label"], random_state=RANDOM_SEED
    )
    sample_size = min(SAMPLE_SIZE, len(train_df_work) // 2 * 2)
else:
    train_df_work = train_df
    sample_size = SAMPLE_SIZE

In [ ]:
if os.path.exists(PARTITIONS_PATH) and not LOCAL_SMOKE_TEST:
    print(f"{PARTITIONS_PATH} ya existe. No se regenera (protocolo fijo).")
    partitions_df = pd.read_csv(PARTITIONS_PATH)
    sample_indices = partitions_df["index"].values
    sample = train_df.loc[train_df["original_index"].isin(sample_indices)].copy()
else:
    sample = train_df_work.sample(
        n=sample_size, stratify=train_df_work["label"], random_state=RANDOM_SEED
    )

    skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_SEED)
    folds = list(skf.split(sample, sample["label"]))

    fold_assignment = pd.Series(index=sample.index, dtype="int64")
    for fold_k, (_, val_idx) in enumerate(folds):
        fold_assignment.iloc[val_idx] = fold_k

    partitions_df = pd.DataFrame({
        "index": sample["original_index"].values,
        "fold": fold_assignment.values,
    }).sort_values("index").reset_index(drop=True)

    assert partitions_df["index"].is_unique, "Hay indices duplicados en partitions.csv"
    assert len(partitions_df) == sample_size, "El tamano de partitions.csv no coincide con sample_size"

    if not LOCAL_SMOKE_TEST:
        partitions_df.to_csv(PARTITIONS_PATH, index=False)
        print(f"Generado {PARTITIONS_PATH} con {len(partitions_df)} filas.")
    else:
        print("[SMOKE TEST] partitions.csv no se persiste en modo prueba local.")

partitions_df.head()

In [ ]:
def build_folds_from_partitions(full_train_df, partitions_df, n_folds=CV_FOLDS):
    """Reconstruye el sample y la asignacion de folds a partir de partitions.csv,
    de forma reproducible para cualquier integrante/seccion posterior."""
    merged = full_train_df.merge(
        partitions_df, left_on="original_index", right_on="index", how="inner"
    )
    merged = merged.sort_values("index").reset_index(drop=True)
    return merged


sample_with_folds = build_folds_from_partitions(train_df, partitions_df)
print(sample_with_folds["fold"].value_counts())
sample_with_folds.head()

## 2. Run de protocolo

**Ejecutar primero, una sola vez entre todos.** Registra los parametros del
muestreo/particionado y los artefactos `protocol/partitions.csv` y
`protocol/members.csv`.

In [ ]:
# encabezado: member_id,notebook_arn
# PENDIENTE: completar con los member_id y notebook_arn reales de los tres integrantes
# una vez confirmados con el profesor.
MEMBERS = [
    {"member_id": "TODO_MEMBER_1", "notebook_arn": "TODO_ARN_1"},
    {"member_id": "TODO_MEMBER_2", "notebook_arn": "TODO_ARN_2"},
    {"member_id": "TODO_MEMBER_3", "notebook_arn": "TODO_ARN_3"},
]

members_df = pd.DataFrame(MEMBERS)
members_path = "protocol/members.csv"
members_df.to_csv(members_path, index=False)
members_df

In [ ]:
def run_protocol():
    if NOTEBOOK_ARN is None:
        print("No hay NOTEBOOK_ARN (no estamos en SageMaker). No se ejecuta el run de protocolo.")
        return None

    with mlflow.start_run(run_name="protocol") as run:
        mlflow.set_tags({
            "lab_run_type": "protocol",
            "notebook_arn": NOTEBOOK_ARN,
        })
        mlflow.log_params({
            "dataset_id": DATASET_ID,
            "dataset_revision": DATASET_REVISION,
            "sampling_strategy": "stratified",
            "sample_size": SAMPLE_SIZE,
            "random_seed": RANDOM_SEED,
            "cv_strategy": "StratifiedKFold",
            "cv_folds": CV_FOLDS,
            "cv_shuffle": True,
        })
        mlflow.log_artifact(PARTITIONS_PATH, artifact_path="protocol")
        mlflow.log_artifact(members_path, artifact_path="protocol")
        log_provenance()
        print("Run de protocolo:", run.info.run_id)
        return run.info.run_id


# Descomentar UNA sola vez, cuando corresponda a este integrante ejecutar el protocolo.
# PROTOCOL_RUN_ID = run_protocol()
PROTOCOL_RUN_ID = None  # reemplazar por el run_id real del protocolo ya ejecutado

## Utilidades compartidas

Funciones de preprocesamiento, construccion de `configuration.json`, entrenamiento
por fold y logging MLflow, reutilizadas en las secciones 3 a 9.

In [ ]:
import re
import string

import emoji as emoji_lib

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
MENTION_PATTERN = re.compile(r"@\w+")
WHITESPACE_PATTERN = re.compile(r"\s+")
ELONGATION_PATTERN = re.compile(r"(.)\1{2,}")

NEGATION_WORDS = {"no", "not", "never", "n't", "cannot", "cant", "dont", "wont", "isnt"}


def build_preprocessor(config):
    """Devuelve una funcion text -> text aplicando las reglas de `config` (dict
    'preprocessing' del esquema de configuration.json)."""

    def preprocess(text):
        t = text
        if config.get("lowercase", True):
            t = t.lower()

        url_mode = config.get("url", "keep")
        if url_mode == "drop":
            t = URL_PATTERN.sub("", t)
        elif url_mode.startswith("token:"):
            t = URL_PATTERN.sub(url_mode.split("token:", 1)[1], t)

        mention_mode = config.get("mention", "keep")
        if mention_mode == "drop":
            t = MENTION_PATTERN.sub("", t)
        elif mention_mode.startswith("token:"):
            t = MENTION_PATTERN.sub(mention_mode.split("token:", 1)[1], t)

        elongation_mode = config.get("elongation", "keep")
        if elongation_mode == "normalize":
            spec = config.get("elongation_spec") or {}
            max_repeat = spec.get("max_repeat", 2)
            t = ELONGATION_PATTERN.sub(lambda m: m.group(1) * max_repeat, t)

        emoji_mode = config.get("emoji", "keep")
        if emoji_mode == "text":
            t = emoji_lib.demojize(t, delimiters=(" :", ": "))

        if config.get("whitespace", "normalize") == "normalize":
            t = WHITESPACE_PATTERN.sub(" ", t).strip()

        stopwords_mode = config.get("stopwords", "keep")
        if stopwords_mode != "keep":
            from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

            negators = set(config.get("negators", []))
            tokens = t.split()
            if stopwords_mode == "remove":
                tokens = [tok for tok in tokens if tok not in ENGLISH_STOP_WORDS]
            elif stopwords_mode == "remove_preserve_negation":
                tokens = [
                    tok for tok in tokens
                    if tok not in ENGLISH_STOP_WORDS or tok in negators
                ]
            t = " ".join(tokens)

        if config.get("lemmatize", False):
            import spacy

            nlp = get_spacy_model()
            doc = nlp(t)
            t = " ".join(tok.lemma_ for tok in doc)

        return t

    return preprocess


_SPACY_MODEL_CACHE = {}


def get_spacy_model(model_name="en_core_web_sm"):
    if model_name not in _SPACY_MODEL_CACHE:
        import spacy

        _SPACY_MODEL_CACHE[model_name] = spacy.load(
            model_name, disable=["parser", "ner"]
        )
    return _SPACY_MODEL_CACHE[model_name]

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class TextPreprocessor(BaseEstimator, TransformerMixin):
    """Wrapper sklearn-compatible alrededor de build_preprocessor, para usar
    dentro de un Pipeline y persistirlo junto con vectorizador + clasificador."""

    def __init__(self, config=None):
        self.config = config or {}

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        preprocess = build_preprocessor(self.config)
        return [preprocess(t) for t in X]


class SpacyEmbeddingVectorizer(BaseEstimator, TransformerMixin):
    """Vectorizador basado en embeddings de spaCy (document_vector_method='mean')."""

    def __init__(self, model_name="en_core_web_md"):
        self.model_name = model_name

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        nlp = get_spacy_model(self.model_name)
        return np.vstack([nlp(t).vector for t in X])

In [ ]:
def build_representation(config):
    """Construye el vectorizador segun el dict 'representation' del esquema."""
    from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

    rep_type = config["type"]
    params = config.get("parameters", {})

    if rep_type == "bow":
        return CountVectorizer(ngram_range=tuple(config["ngram_range"]), **params)
    if rep_type == "tfidf":
        return TfidfVectorizer(ngram_range=tuple(config["ngram_range"]), **params)
    if rep_type == "spacy_embedding":
        return SpacyEmbeddingVectorizer(model_name=config["spacy_model"])
    raise ValueError(f"representation.type desconocido: {rep_type}")


def build_classifier(config):
    """Construye el clasificador segun el dict 'classifier' del esquema."""
    from sklearn.dummy import DummyClassifier
    from sklearn.linear_model import LogisticRegression, SGDClassifier
    from sklearn.svm import LinearSVC

    clf_type = config["type"]
    params = config.get("parameters", {})

    if clf_type == "most_frequent":
        return DummyClassifier(strategy="most_frequent")
    if clf_type == "logistic_regression":
        return LogisticRegression(**{"max_iter": 1000, **params})
    if clf_type == "linear_svm":
        return LinearSVC(**params)
    if clf_type == "sgd":
        return SGDClassifier(**params)
    raise ValueError(f"classifier.type desconocido: {clf_type}")

In [ ]:
import sklearn
import spacy as spacy_module


def build_pipeline_from_config(config):
    """Construye el Pipeline sklearn completo (preprocesamiento + representacion +
    clasificador) a partir de un dict con el esquema de configuration.json."""
    from sklearn.pipeline import Pipeline

    steps = []
    if config["preprocessing"] is not None:
        steps.append(("preprocess", TextPreprocessor(config["preprocessing"])))
    if config["representation"] is not None:
        steps.append(("vectorize", build_representation(config["representation"])))
    steps.append(("classify", build_classifier(config["classifier"])))
    return Pipeline(steps)


def default_preprocessing_config(**overrides):
    base = {
        "lowercase": True,
        "url": "token:url",
        "mention": "token:user",
        "whitespace": "normalize",
        "stopwords": "keep",
        "negators": [],
        "lemmatize": False,
        "elongation": "keep",
        "elongation_spec": None,
        "emoji": "keep",
        "emoji_spec": None,
        "resources": {},
        "additional": {},
    }
    base.update(overrides)
    return base


def default_representation_config(**overrides):
    base = {
        "type": "bow",
        "ngram_range": [1, 1],
        "library": "sklearn",
        "library_version": sklearn.__version__,
        "spacy_model": None,
        "spacy_model_version": None,
        "parameters": {},
    }
    base.update(overrides)
    return base


def default_classifier_config(**overrides):
    base = {
        "type": "logistic_regression",
        "library": "sklearn",
        "library_version": sklearn.__version__,
        "parameters": {},
    }
    base.update(overrides)
    return base

In [ ]:
from sklearn.metrics import f1_score


def evaluate_cv(config, sample_with_folds, text_col="text", label_col="label"):
    """Entrena y evalua el pipeline definido por `config` en los 3 folds del
    protocolo. Devuelve (macro_f1_por_fold, macro_f1_mean, macro_f1_std)."""
    fold_scores = []
    for fold_k in sorted(sample_with_folds["fold"].unique()):
        val_mask = sample_with_folds["fold"] == fold_k
        train_part = sample_with_folds.loc[~val_mask]
        val_part = sample_with_folds.loc[val_mask]

        pipeline = build_pipeline_from_config(config)
        pipeline.fit(train_part[text_col], train_part[label_col])
        preds = pipeline.predict(val_part[text_col])

        score = f1_score(val_part[label_col], preds, average="macro")
        fold_scores.append(score)

    fold_scores = np.array(fold_scores)
    return fold_scores, fold_scores.mean(), fold_scores.std(ddof=0)

In [ ]:
CONFIGURATION_SCHEMA_KEYS = {"preprocessing", "representation", "classifier"}


def save_configuration_json(config, path="run/configuration.json"):
    assert set(config.keys()) == CONFIGURATION_SCHEMA_KEYS, (
        f"configuration.json debe tener exactamente las claves {CONFIGURATION_SCHEMA_KEYS}"
    )
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        json.dump(config, f, indent=2)
    return path


def log_experiment_run(
    run_name,
    config,
    lab_experiment_id,
    lab_stage,
    lab_configuration_id,
    fold_scores,
    macro_f1_mean,
    macro_f1_std,
    extra_tags=None,
    extra_metrics=None,
    extra_params=None,
):
    """Registra un run experimental completo: tags, params, metricas y artefactos
    obligatorios segun CLAUDE.md."""
    if NOTEBOOK_ARN is None:
        raise RuntimeError(
            "No hay NOTEBOOK_ARN: este run no seria valido para la entrega. "
            "Ejecutar desde SageMaker."
        )
    if PROTOCOL_RUN_ID is None:
        raise RuntimeError("Falta PROTOCOL_RUN_ID: asignar el run_id del run de protocolo.")
    if LAB_MEMBER_ID.startswith("TODO"):
        raise RuntimeError("Falta confirmar LAB_MEMBER_ID con el profesor antes de loggear runs.")

    config_path = save_configuration_json(config)

    with mlflow.start_run(run_name=run_name) as run:
        tags = {
            "lab_run_type": "experiment",
            "lab_protocol_run_id": PROTOCOL_RUN_ID,
            "lab_experiment_id": lab_experiment_id,
            "lab_stage": lab_stage,
            "lab_member_id": LAB_MEMBER_ID,
            "lab_configuration_id": lab_configuration_id,
            "notebook_arn": NOTEBOOK_ARN,
        }
        if extra_tags:
            tags.update(extra_tags)
        mlflow.set_tags(tags)

        if extra_params:
            mlflow.log_params(extra_params)

        metrics = {
            "macro_f1_fold_0": float(fold_scores[0]),
            "macro_f1_fold_1": float(fold_scores[1]),
            "macro_f1_fold_2": float(fold_scores[2]),
            "macro_f1_mean": float(macro_f1_mean),
            "macro_f1_std": float(macro_f1_std),
        }
        if extra_metrics:
            metrics.update(extra_metrics)
        mlflow.log_metrics(metrics)

        mlflow.log_artifact(config_path, artifact_path="run")
        log_provenance()

        print(f"Run '{run_name}' registrado: {run.info.run_id} | macro_f1_mean={macro_f1_mean:.4f}")
        return run.info.run_id

## 3. T0 — baseline trivial

Un integrante lo ejecuta una sola vez; no cuenta para el minimo individual.
`preprocessing=null`, `representation=null`, `classifier.type=most_frequent`.

In [ ]:
t0_config = {
    "preprocessing": None,
    "representation": None,
    "classifier": default_classifier_config(type="most_frequent"),
}

# t0_fold_scores, t0_mean, t0_std = evaluate_cv(t0_config, sample_with_folds)
# t0_run_id = log_experiment_run(
#     run_name="T0",
#     config=t0_config,
#     lab_experiment_id="T0",
#     lab_stage="reference",
#     lab_configuration_id="CFG_T0",
#     fold_scores=t0_fold_scores,
#     macro_f1_mean=t0_mean,
#     macro_f1_std=t0_std,
# )

## 4. B0 — baseline experimental

Pipeline simple pero no trivial: BOW unigrama + Logistic Regression, sin
preprocesamiento adicional mas alla de las normalizaciones basicas. Un
integrante lo ejecuta una sola vez; no cuenta para el minimo individual.

In [ ]:
b0_config = {
    "preprocessing": default_preprocessing_config(),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="logistic_regression"),
}

# b0_fold_scores, b0_mean, b0_std = evaluate_cv(b0_config, sample_with_folds)
# b0_run_id = log_experiment_run(
#     run_name="B0",
#     config=b0_config,
#     lab_experiment_id="B0",
#     lab_stage="baseline",
#     lab_configuration_id="CFG_B0",
#     fold_scores=b0_fold_scores,
#     macro_f1_mean=b0_mean,
#     macro_f1_std=b0_std,
# )

## 5. Preprocesamiento

**Runs asignados a MEMBER_ID_AQUI** (`lab_member_id` = el confirmado con el profesor):
`P_STOPWORDS`, `P_STOPWORDS_NEGATION`, `P_LEMMA`, `P_ELONGATION`, `P_EMOJI`.

Cada experimento parte de B0 y cambia una sola decision de preprocesamiento a la vez,
manteniendo representacion y clasificador fijos en los valores de B0.

In [ ]:
# P_STOPWORDS: eliminar stopwords sin preservar negacion
p_stopwords_config = {
    "preprocessing": default_preprocessing_config(stopwords="remove"),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="logistic_regression"),
}

# fold_scores, mean_, std_ = evaluate_cv(p_stopwords_config, sample_with_folds)
# log_experiment_run(
#     run_name="P_STOPWORDS",
#     config=p_stopwords_config,
#     lab_experiment_id="P_STOPWORDS",
#     lab_stage="preprocessing",
#     lab_configuration_id="CFG_P_STOPWORDS",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

In [ ]:
# P_STOPWORDS_NEGATION: eliminar stopwords preservando negadores
p_stopwords_negation_config = {
    "preprocessing": default_preprocessing_config(
        stopwords="remove_preserve_negation",
        negators=["no", "not", "never", "n't", "cannot", "cant", "dont", "wont", "isnt"],
    ),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="logistic_regression"),
}

# fold_scores, mean_, std_ = evaluate_cv(p_stopwords_negation_config, sample_with_folds)
# log_experiment_run(
#     run_name="P_STOPWORDS_NEGATION",
#     config=p_stopwords_negation_config,
#     lab_experiment_id="P_STOPWORDS_NEGATION",
#     lab_stage="preprocessing",
#     lab_configuration_id="CFG_P_STOPWORDS_NEGATION",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

In [ ]:
# P_LEMMA: lematizacion con spaCy
p_lemma_config = {
    "preprocessing": default_preprocessing_config(lemmatize=True),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="logistic_regression"),
}

# fold_scores, mean_, std_ = evaluate_cv(p_lemma_config, sample_with_folds)
# log_experiment_run(
#     run_name="P_LEMMA",
#     config=p_lemma_config,
#     lab_experiment_id="P_LEMMA",
#     lab_stage="preprocessing",
#     lab_configuration_id="CFG_P_LEMMA",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

In [ ]:
# P_ELONGATION: normalizar alargamientos ("looove" -> "loove")
p_elongation_config = {
    "preprocessing": default_preprocessing_config(
        elongation="normalize", elongation_spec={"max_repeat": 2},
    ),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="logistic_regression"),
}

# fold_scores, mean_, std_ = evaluate_cv(p_elongation_config, sample_with_folds)
# log_experiment_run(
#     run_name="P_ELONGATION",
#     config=p_elongation_config,
#     lab_experiment_id="P_ELONGATION",
#     lab_stage="preprocessing",
#     lab_configuration_id="CFG_P_ELONGATION",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

In [ ]:
# P_EMOJI: convertir emojis a texto (":heart:", etc.)
p_emoji_config = {
    "preprocessing": default_preprocessing_config(
        emoji="text", emoji_spec={"library": "emoji", "delimiters": [" :", ": "]},
    ),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="logistic_regression"),
}

# fold_scores, mean_, std_ = evaluate_cv(p_emoji_config, sample_with_folds)
# log_experiment_run(
#     run_name="P_EMOJI",
#     config=p_emoji_config,
#     lab_experiment_id="P_EMOJI",
#     lab_stage="preprocessing",
#     lab_configuration_id="CFG_P_EMOJI",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

## 6. Representacion

**Runs asignados al Integrante 2**: `R_BOW`, `R_TFIDF_UNI`, `R_TFIDF_UNI_BI`, `R_SPACY`.

Cada experimento parte de B0 y cambia solo la representacion, manteniendo el
preprocesamiento y el clasificador de B0.

In [ ]:
r_bow_config = {
    "preprocessing": default_preprocessing_config(),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="logistic_regression"),
}
# (equivalente a B0; se deja explicito para claridad de la matriz de experimentos)
# fold_scores, mean_, std_ = evaluate_cv(r_bow_config, sample_with_folds)
# log_experiment_run(
#     run_name="R_BOW", config=r_bow_config,
#     lab_experiment_id="R_BOW", lab_stage="representation",
#     lab_configuration_id="CFG_R_BOW",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

In [ ]:
r_tfidf_uni_config = {
    "preprocessing": default_preprocessing_config(),
    "representation": default_representation_config(type="tfidf", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="logistic_regression"),
}
# fold_scores, mean_, std_ = evaluate_cv(r_tfidf_uni_config, sample_with_folds)
# log_experiment_run(
#     run_name="R_TFIDF_UNI", config=r_tfidf_uni_config,
#     lab_experiment_id="R_TFIDF_UNI", lab_stage="representation",
#     lab_configuration_id="CFG_R_TFIDF_UNI",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

In [ ]:
r_tfidf_uni_bi_config = {
    "preprocessing": default_preprocessing_config(),
    "representation": default_representation_config(type="tfidf", ngram_range=[1, 2]),
    "classifier": default_classifier_config(type="logistic_regression"),
}
# fold_scores, mean_, std_ = evaluate_cv(r_tfidf_uni_bi_config, sample_with_folds)
# log_experiment_run(
#     run_name="R_TFIDF_UNI_BI", config=r_tfidf_uni_bi_config,
#     lab_experiment_id="R_TFIDF_UNI_BI", lab_stage="representation",
#     lab_configuration_id="CFG_R_TFIDF_UNI_BI",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

In [ ]:
r_spacy_config = {
    "preprocessing": default_preprocessing_config(),
    "representation": default_representation_config(
        type="spacy_embedding",
        ngram_range=[],
        spacy_model="en_core_web_md",
        spacy_model_version=spacy_module.__version__,
        parameters={"document_vector_method": "mean"},
    ),
    "classifier": default_classifier_config(type="logistic_regression"),
}
# fold_scores, mean_, std_ = evaluate_cv(r_spacy_config, sample_with_folds)
# log_experiment_run(
#     run_name="R_SPACY", config=r_spacy_config,
#     lab_experiment_id="R_SPACY", lab_stage="representation",
#     lab_configuration_id="CFG_R_SPACY",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

## 7. Clasificador

**Runs asignados al Integrante 3**: `C_LOGREG`, `C_LINEAR_SVM`, `C_SGD`.

Cada experimento parte de B0 y cambia solo el clasificador, manteniendo el
preprocesamiento y la representacion de B0.

In [ ]:
c_logreg_config = {
    "preprocessing": default_preprocessing_config(),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="logistic_regression"),
}
# (equivalente a B0)
# fold_scores, mean_, std_ = evaluate_cv(c_logreg_config, sample_with_folds)
# log_experiment_run(
#     run_name="C_LOGREG", config=c_logreg_config,
#     lab_experiment_id="C_LOGREG", lab_stage="classifier",
#     lab_configuration_id="CFG_C_LOGREG",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

In [ ]:
c_linear_svm_config = {
    "preprocessing": default_preprocessing_config(),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="linear_svm"),
}
# fold_scores, mean_, std_ = evaluate_cv(c_linear_svm_config, sample_with_folds)
# log_experiment_run(
#     run_name="C_LINEAR_SVM", config=c_linear_svm_config,
#     lab_experiment_id="C_LINEAR_SVM", lab_stage="classifier",
#     lab_configuration_id="CFG_C_LINEAR_SVM",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

In [ ]:
c_sgd_config = {
    "preprocessing": default_preprocessing_config(),
    "representation": default_representation_config(type="bow", ngram_range=[1, 1]),
    "classifier": default_classifier_config(type="sgd"),
}
# fold_scores, mean_, std_ = evaluate_cv(c_sgd_config, sample_with_folds)
# log_experiment_run(
#     run_name="C_SGD", config=c_sgd_config,
#     lab_experiment_id="C_SGD", lab_stage="classifier",
#     lab_configuration_id="CFG_C_SGD",
#     fold_scores=fold_scores, macro_f1_mean=mean_, macro_f1_std=std_,
# )

## 8. Seleccion del pipeline candidato

Comparar `macro_f1_mean` de todos los runs registrados (`lab_run_type=experiment`)
en el experimento, **sin tocar test**, y elegir la combinacion de preprocesamiento +
representacion + clasificador con mejor desempeno/robustez (menor `macro_f1_std`
como criterio de desempate).

In [ ]:
def fetch_experiment_runs():
    experiment = client.get_experiment_by_name(EXPERIMENT_NAME)
    return client.search_runs(
        experiment_ids=[experiment.experiment_id],
        filter_string="tags.lab_run_type = 'experiment'",
        max_results=5000,
    )


# runs = fetch_experiment_runs()
# comparison_df = pd.DataFrame([
#     {
#         "run_id": r.info.run_id,
#         "lab_experiment_id": r.data.tags.get("lab_experiment_id"),
#         "lab_stage": r.data.tags.get("lab_stage"),
#         "macro_f1_mean": r.data.metrics.get("macro_f1_mean"),
#         "macro_f1_std": r.data.metrics.get("macro_f1_std"),
#     }
#     for r in runs
# ]).sort_values("macro_f1_mean", ascending=False)
# comparison_df

In [ ]:
# Una vez decidido el candidato (a partir de la tabla anterior), fijar aqui su
# configuracion y el run_id seleccionado para las siguientes secciones.
candidate_config = None  # dict con el esquema completo del pipeline elegido
CANDIDATE_RUN_ID = None  # run_id del experimento seleccionado como candidato

## 9. Ablacion

Revertir, una a la vez, las decisiones en las que el candidato difiere de B0.
Si difiere en 1 decision, evaluar 1 ablacion; si difiere en 2 o mas, evaluar
al menos 2.

In [ ]:
ABLATABLE_DECISIONS = {
    "preprocessing.stopwords",
    "preprocessing.lemmatize",
    "preprocessing.elongation",
    "preprocessing.emoji",
    "representation",
    "classifier",
}


def revert_decision(config, decision, b0_config=b0_config):
    """Devuelve una copia de `config` con la decision indicada revertida al valor de B0."""
    reverted = json.loads(json.dumps(config))  # deep copy
    if decision == "preprocessing.stopwords":
        reverted["preprocessing"]["stopwords"] = b0_config["preprocessing"]["stopwords"]
        reverted["preprocessing"]["negators"] = b0_config["preprocessing"]["negators"]
    elif decision == "preprocessing.lemmatize":
        reverted["preprocessing"]["lemmatize"] = b0_config["preprocessing"]["lemmatize"]
    elif decision == "preprocessing.elongation":
        reverted["preprocessing"]["elongation"] = b0_config["preprocessing"]["elongation"]
        reverted["preprocessing"]["elongation_spec"] = b0_config["preprocessing"]["elongation_spec"]
    elif decision == "preprocessing.emoji":
        reverted["preprocessing"]["emoji"] = b0_config["preprocessing"]["emoji"]
        reverted["preprocessing"]["emoji_spec"] = b0_config["preprocessing"]["emoji_spec"]
    elif decision == "representation":
        reverted["representation"] = b0_config["representation"]
    elif decision == "classifier":
        reverted["classifier"] = b0_config["classifier"]
    else:
        raise ValueError(f"Decision ablacionable desconocida: {decision}")
    return reverted


def run_ablation(decision, candidate_config, candidate_mean, lab_configuration_id):
    ablated_config = revert_decision(candidate_config, decision)
    fold_scores, mean_, std_ = evaluate_cv(ablated_config, sample_with_folds)
    macro_f1_delta = candidate_mean - mean_

    return log_experiment_run(
        run_name=f"ABLATION_{decision}",
        config=ablated_config,
        lab_experiment_id="ABLATION",
        lab_stage="ablation",
        lab_configuration_id=lab_configuration_id,
        fold_scores=fold_scores,
        macro_f1_mean=mean_,
        macro_f1_std=std_,
        extra_tags={"lab_ablation_parent_run_id": CANDIDATE_RUN_ID},
        extra_params={"ablation_reverted_decision": decision},
        extra_metrics={"macro_f1_delta": float(macro_f1_delta)},
    )


# Ejemplo de uso una vez definido candidate_config y su macro_f1_mean:
# run_ablation("preprocessing.stopwords", candidate_config, candidate_macro_f1_mean, "CFG_ABLATION_1")

## 10. Reentrenamiento final + evaluacion en test

Se ejecuta **una sola vez**, con la configuracion candidata ya seleccionada
(sin ninguna decision tomada a partir del resultado en test).

In [ ]:
def train_final_model(config, train_df, text_col="text", label_col="label"):
    pipeline = build_pipeline_from_config(config)
    pipeline.fit(train_df[text_col], train_df[label_col])
    return pipeline


def to_sentiment_label(pred):
    return "positive" if int(pred) == 1 else "negative"


class SentimentPipelineWrapper(BaseEstimator, TransformerMixin):
    """Envuelve el pipeline entrenado para que `predict` devuelva 'positive'/'negative'
    directamente, recibiendo texto crudo como entrada."""

    def __init__(self, inner_pipeline):
        self.inner_pipeline = inner_pipeline

    def fit(self, X, y=None):
        return self

    def predict(self, X):
        raw = self.inner_pipeline.predict(X)
        return [to_sentiment_label(p) for p in raw]

In [ ]:
# final_config = candidate_config  # ya seleccionado en la seccion 8
# final_pipeline_raw = train_final_model(final_config, train_df)
# final_pipeline = SentimentPipelineWrapper(final_pipeline_raw)
#
# test_df = ds["test"].to_pandas()
# test_preds = final_pipeline.predict(test_df["text"])
# test_true = test_df["label"].map(to_sentiment_label)
#
# from sklearn.metrics import f1_score as f1_score_final
# test_macro_f1 = f1_score_final(
#     test_true, test_preds, average="macro", labels=["negative", "positive"]
# )
# print("test_macro_f1:", test_macro_f1)

In [ ]:
# with mlflow.start_run(run_name="FINAL") as final_run:
#     mlflow.set_tags({
#         "lab_run_type": "final",
#         "lab_protocol_run_id": PROTOCOL_RUN_ID,
#         "lab_selected_experiment_run_id": CANDIDATE_RUN_ID,
#         "lab_configuration_id": "CFG_FINAL",
#         "lab_member_id": LAB_MEMBER_ID,
#         "notebook_arn": NOTEBOOK_ARN,
#     })
#     mlflow.log_param("training_size", 1_360_000)
#     mlflow.log_metric("test_macro_f1", test_macro_f1)
#     mlflow.sklearn.log_model(final_pipeline, "model")
#     config_path = save_configuration_json(final_config)
#     mlflow.log_artifact(config_path, artifact_path="run")
#     log_provenance()
#
#     FINAL_RUN_ID = final_run.info.run_id
#
# mv = mlflow.register_model(f"runs:/{FINAL_RUN_ID}/model", "sentiment140")
# client.set_registered_model_alias("sentiment140", "champion", mv.version)
# print("Modelo final registrado como sentiment140@champion, version", mv.version)

## 11. Analisis de errores

Semilla 42, minimo 20 errores, ambas clases si existen. Genera
`reports/error_analysis.csv` y `reports/error_analysis.md`, que tambien se
registran como artefactos del run final.

In [ ]:
ERROR_ANALYSIS_SEED = 42
MIN_ERRORS = 20

ERROR_CATEGORIES = [
    "negation", "intensification", "contrast", "mixed", "emoji",
    "elongation", "informal", "hashtag", "sarcasm", "other",
]


def build_error_analysis(test_df, test_preds, test_true, min_errors=MIN_ERRORS, seed=ERROR_ANALYSIS_SEED):
    """test_df debe conservar su indice posicional original del split test (0..N-1);
    ese indice, no el de train_df, es el que va en la columna 'index' del CSV."""
    errors_mask = pd.Series(test_preds).values != pd.Series(test_true).values
    errors_df = test_df.loc[errors_mask].copy()
    errors_df["true_label"] = pd.Series(test_true).loc[errors_mask].values
    errors_df["predicted_label"] = pd.Series(test_preds).loc[errors_mask].values

    n_per_class = max(min_errors // 2, 1)
    sampled_parts = []
    for cls in errors_df["true_label"].unique():
        cls_errors = errors_df[errors_df["true_label"] == cls]
        sampled_parts.append(
            cls_errors.sample(n=min(n_per_class, len(cls_errors)), random_state=seed)
        )
    sampled = pd.concat(sampled_parts)

    if len(sampled) < min_errors:
        remaining = errors_df.drop(sampled.index)
        extra_needed = min_errors - len(sampled)
        if len(remaining) > 0:
            extra = remaining.sample(n=min(extra_needed, len(remaining)), random_state=seed)
            sampled = pd.concat([sampled, extra])

    sampled = sampled.reset_index().rename(columns={"index": "index"})
    # Categoria: placeholder "other"; requiere revision manual/heuristica por integrante.
    sampled["category"] = "other"

    result = sampled[["index", "text", "true_label", "predicted_label", "category"]]
    return result


# test_df = ds["test"].to_pandas()
# test_df debe conservar su indice posicional (0..N-1) tal cual lo entrega el split test.
# error_analysis_df = build_error_analysis(test_df, test_preds, test_true)
# os.makedirs("reports", exist_ok=True)
# error_analysis_df.to_csv("reports/error_analysis.csv", index=False)
# error_analysis_df.head()

In [ ]:
def build_error_analysis_report(error_analysis_df, path="reports/error_analysis.md"):
    freq = error_analysis_df["category"].value_counts()
    top_two = freq.head(2)

    lines = ["# Analisis de errores\n", "\n", "## Frecuencia por categoria\n", "\n"]
    lines.append("| Categoria | Frecuencia |\n|---|---|\n")
    for cat, count in freq.items():
        lines.append(f"| {cat} | {count} |\n")

    lines.append("\n## Interpretacion de los dos patrones mas frecuentes\n\n")
    for cat, count in top_two.items():
        lines.append(f"### {cat} ({count} casos)\n\n")
        lines.append("TODO: interpretar por que el modelo falla en estos casos, con ejemplos concretos.\n\n")

    with open(path, "w") as f:
        f.writelines(lines)
    return path


# build_error_analysis_report(error_analysis_df)